In [1]:
import pandas as pd

### cleaned parquet load karne kai liye

In [2]:
orders = pd.read_parquet("../processed/orders.parquet")
op = pd.read_parquet("../processed/order_products.parquet")
products = pd.read_parquet("../processed/products.parquet")
print(orders.shape, op.shape, products.shape)

(3346083, 8) (33819106, 4) (49688, 6)


### har product ke orders aur reorders ginti karne kai liye

In [3]:
prod_stats = op.groupby("product_id")["reordered"].agg(orders="count", reorders="sum").reset_index()
prod_stats = prod_stats.merge(products, on="product_id")
prod_stats["reorder_rate"] = prod_stats["reorders"] / prod_stats["orders"]
prod_stats.shape

(49685, 9)

### sabse zyada bikne wale 10 products 

In [4]:
prod_stats.sort_values("orders", ascending=False).head(10)[["product_name", "department", "orders", "reorder_rate"]]

,product_name,department,orders,reorder_rate
24849,Banana,produce,491291,0.845051
13173,Bag of Organic Bananas,produce,394930,0.833755
21134,Organic Strawberries,produce,275577,0.778178
21900,Organic Baby Spinach,produce,251705,0.774474
47205,Organic Hass Avocado,produce,220877,0.797607
47762,Organic Avocado,produce,184224,0.761410
47622,Large Lemon,produce,160792,0.697659
16794,Strawberries,produce,149445,0.699843
26206,Limes,produce,146660,0.681863
27842,Organic Whole Milk,dairy eggs,142813,0.831045


In [5]:
never = products[~products["product_id"].isin(op["product_id"])]
never[["product_id", "product_name", "department"]]

,product_id,product_name,department
3629,3630,Protein Granola Apple Crisp,breakfast
7044,7045,Unpeeled Apricot Halves in Heavy Syrup,pantry
46624,46625,Single Barrel Kentucky Straight Bourbon Whiskey,beverages


In [6]:
dept_stats = prod_stats.groupby("department").agg(
    products=("product_id", "count"),
    orders=("orders", "sum"),
    reorders=("reorders", "sum"),
).reset_index()
dept_stats["reorder_rate"] = dept_stats["reorders"] / dept_stats["orders"]
dept_stats.sort_values("orders", ascending=False)

,department,products,orders,reorders,reorder_rate
19,produce,1684,9888378,6432596,0.650521
7,dairy eggs,3449,5631067,3773723,0.670161
20,snacks,6264,3006412,1727075,0.574464
3,beverages,4364,2804175,1832952,0.653651
10,frozen,4007,2336858,1268058,0.542634
16,pantry,5370,1956819,679799,0.347400
2,bakery,1516,1225181,769880,0.628381
6,canned goods,2092,1114857,511317,0.458639
8,deli,1322,1095540,666231,0.608130
9,dry goods pasta,1858,905340,418466,0.462220


In [7]:
basket = op.groupby("order_id").agg(
    basket_size=("product_id", "count"),
    reordered_items=("reordered", "sum"),
).reset_index()
orders_enriched = orders.merge(basket, on="order_id")
orders_enriched.shape

(3346083, 10)

In [8]:
users = orders_enriched.groupby("user_id").agg(
    total_orders=("order_id", "count"),
    avg_gap_days=("days_since_prior_order", "mean"),
    avg_basket_size=("basket_size", "mean"),
    total_items=("basket_size", "sum"),
    reordered_items=("reordered_items", "sum"),
).reset_index()
users["reorder_ratio"] = users["reordered_items"] / users["total_items"]
users.shape

(206209, 7)

In [9]:
users.describe()

,user_id,total_orders,avg_gap_days,avg_basket_size,total_items,reordered_items,reorder_ratio
count,206209.000000,206209.000000,206209.000000,206209.000000,206209.000000,206209.000000,206209.000000
mean,103105.000000,16.226658,15.360843,9.984527,164.004025,96.772498,0.444332
std,59527.555167,16.662238,6.985487,5.840846,206.014923,160.293373,0.206270
min,1.000000,3.000000,0.000000,1.000000,3.000000,0.000000,0.000000
25%,51553.000000,6.000000,9.666667,5.777778,44.000000,13.000000,0.285714
50%,103105.000000,10.000000,14.800000,9.000000,90.000000,37.000000,0.442623
75%,154657.000000,20.000000,20.500000,13.000000,196.000000,106.000000,0.601562
max,206209.000000,100.000000,30.000000,60.500000,3725.000000,3275.000000,0.989637


In [10]:
orders_enriched.to_parquet("../processed/orders_enriched.parquet", index=False)
users.to_parquet("../processed/users_summary.parquet", index=False)
prod_stats.to_parquet("../processed/product_summary.parquet", index=False)
dept_stats.to_parquet("../processed/department_summary.parquet", index=False)